In [ ]:
#!pip install -U unsloth_zoo

!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes
!pip install datasets transformers

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-7wmv79xb/unsloth_56d120de837b4c22a9bd8f986e500be6
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-7wmv79xb/unsloth_56d120de837b4c22a9bd8f986e500be6
  Resolved https://github.com/unslothai/unsloth.git to commit 4f9c8321a2136e62fd86fe722a544afd534334a5
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for unsloth: filename=unsloth-2026.4.8-py3-none-any.whl size=32125753 sha256=b1b96a3ec74a2dbefe0e073dc3bbb9fd96460e072d6f7a861b36ebe3ced7c891
  Stored in directory: /tmp/pip-ephem-wheel-cache-ntlzglx6/wheels/60/3e/1f/e576c07051d90cf64b6a41434d87ccf4db33fafd5343bf5de0
Successfully built unsloth
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 721.6/721.6 kB 24.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.2 MB/s eta 0:00:00


In [ ]:
!pip install -U unsloth_zoo


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 421.9/421.9 kB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 26.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 52.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 74.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 36.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.0/225.0 kB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 14.9 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Succe

In [ ]:
# finetune_unsloth.py — Fine-tuning with Unsloth (EMR / SageMaker / Local)
# Picks up from preprocessing_final output: s3://20596297-project-data/output/


import os
import torch
from datasets import load_dataset, DatasetDict
import unsloth
from unsloth import FastLanguageModel
from transformers import TrainingArguments
from trl import SFTTrainer
from unsloth.chat_templates import get_chat_template

#  1. Config

MODEL_NAME = "unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit"  # swap freely
MAX_SEQ_LENGTH = 2048          # matches your answer cap (~3000 chars ≈ ~750 tokens)
DTYPE          = None          # auto-detect (bfloat16 on Ampere+, float16 otherwise)
LOAD_IN_4BIT   = True          # QLoRA — keeps VRAM low

# LoRA hyperparams
LORA_R         = 16
LORA_ALPHA     = 32
LORA_DROPOUT   = 0.05
TARGET_MODULES = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj",
]

# Training hyperparams
EPOCHS         = 3
BATCH_SIZE     = 4             # per device
GRAD_ACCUM     = 4             # effective batch = 16
LR             = 2e-4
WARMUP_RATIO   = 0.03
LR_SCHEDULER   = "cosine"
MAX_STEPS      = -1            # -1 = run full epochs
SAVE_STEPS     = 500
LOGGING_STEPS  = 50
OUTPUT_DIR     = "/tmp/unsloth-checkpoints"


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [ ]:
import os
os.environ["UNSLOTH_DISABLE_STATS"] = "1"

In [ ]:
# ── 2. Load Model + Tokenizer via Unsloth ─────────────────────
print(f"\n Loading model: {MODEL_NAME}")
import os
os.environ["UNSLOTH_DISABLE_STATS"] = "1"
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name      = MODEL_NAME,
    max_seq_length  = MAX_SEQ_LENGTH,
    dtype           = DTYPE,
    load_in_4bit    = LOAD_IN_4BIT,
)

# Apply chat template (Llama-3 style — change if using Mistral/Qwen/etc.)
tokenizer = get_chat_template(tokenizer, chat_template="llama-3")

# ── 3. Apply LoRA (PEFT) ──────────────────────────────────────
print(" Applying LoRA adapters...")

model = FastLanguageModel.get_peft_model(
    model,
    r                   = LORA_R,
    lora_alpha          = LORA_ALPHA,
    lora_dropout        = LORA_DROPOUT,
    target_modules      = TARGET_MODULES,
    bias                = "none",
    use_gradient_checkpointing = "unsloth",   # saves ~30% VRAM
    random_state        = 42,
    use_rslora          = False,
    loftq_config        = None,
)

model.print_trainable_parameters()


🦥 Loading model: unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit
==((====))==  Unsloth 2026.4.8: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/1.14G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/270 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.


🔧 Applying LoRA adapters...


Unsloth 2026.4.8 patched 28 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820


In [ ]:
from datasets import load_dataset, DatasetDict

print("\n Loading local dataset...")

dataset = DatasetDict({
    "train": load_dataset(
        "csv",
        data_files="data/train.csv",
        split="train",
    ),
    "validation": load_dataset(
        "csv",
        data_files="data/val.csv",
        split="train",
    ),
})

print(f" Train samples   : {len(dataset['train']):,}")
print(f" Val samples     : {len(dataset['validation']):,}")


📦 Loading local dataset...


Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

✅ Train samples   : 92,310
✅ Val samples     : 11,430


In [ ]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    max_steps=MAX_STEPS,

    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,

    learning_rate=LR,
    warmup_ratio=WARMUP_RATIO,
    lr_scheduler_type=LR_SCHEDULER,

    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),

    optim="adamw_8bit",
    weight_decay=0.01,

    logging_steps=LOGGING_STEPS,
    save_steps=SAVE_STEPS,
    save_total_limit=3,

    report_to="none",
    seed=42,
    dataloader_num_workers=4,
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [ ]:
def add_text(example):
    return {
        "text": f"""### Question:
{example['question']}

### Answer:
{example['answer']}"""
    }

dataset = dataset.map(add_text)

Map:   0%|          | 0/92310 [00:00<?, ? examples/s]

Map:   0%|          | 0/11430 [00:00<?, ? examples/s]

In [ ]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,

    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],

    max_seq_length=MAX_SEQ_LENGTH,

    formatting_func=lambda x: x["text"],

    args=training_args,
    packing=True,
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/92310 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/11430 [00:00<?, ? examples/s]

In [ ]:
# ── 9. Save LoRA Adapter locally then push to S3 ──────────────
LOCAL_ADAPTER = "/tmp/lora-adapter"
print(f"\n Saving LoRA adapter to {LOCAL_ADAPTER}...")
model.save_pretrained(LOCAL_ADAPTER)
tokenizer.save_pretrained(LOCAL_ADAPTER)




💾 Saving LoRA adapter to /tmp/lora-adapter...


Unsloth: Restored added_tokens_decoder metadata in /tmp/lora-adapter/tokenizer_config.json.


('/tmp/lora-adapter/tokenizer_config.json',
 '/tmp/lora-adapter/chat_template.jinja',
 '/tmp/lora-adapter/tokenizer.json')

In [ ]:
import os

# ── 1) Save LoRA Adapter ─────────────────────────────
LOCAL_ADAPTER = "./lora-adapter"

print(f" Saving LoRA adapter → {LOCAL_ADAPTER}")
model.save_pretrained(LOCAL_ADAPTER)
tokenizer.save_pretrained(LOCAL_ADAPTER)

print(" Adapter saved.")


# ── 2) Save Merged 16-bit Model ─────────────────────
MERGED_DIR = "./merged-model"

print(f"\n Merging + saving full model → {MERGED_DIR}")

model.save_pretrained_merged(
    MERGED_DIR,
    tokenizer,
    save_method="merged_16bit",
)

print(" Merged model saved.")


# ── 3) Export GGUF (Ollama / llama.cpp) ─────────────
GGUF_DIR = "./gguf-model"

print(f"\n Exporting GGUF → {GGUF_DIR}")

model.save_pretrained_gguf(
    GGUF_DIR,
    tokenizer,
    quantization_method="q4_k_m"
)

print(" GGUF saved.")

💾 Saving LoRA adapter → ./lora-adapter


Unsloth: Restored added_tokens_decoder metadata in ./lora-adapter/tokenizer_config.json.


✅ Adapter saved.

🔀 Merging + saving full model → ./merged-model


config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

Unsloth: Restored added_tokens_decoder metadata in ./merged-model/tokenizer_config.json.


...

Unsloth: Preparing safetensor model files: 100%|██████████| 1/1 [00:32<00:00, 32.38s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:44<00:00, 44.56s/it]


Unsloth: Merge process complete. Saved to `/content/gguf-model`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Updating system package directories
Unsloth: Cloning llama.cpp repository...
Unsloth: Building llama.cpp - please wait 1 to 3 minutes
Unsloth: Successfully installed llama.cpp!
Unsloth: Preparing converter script...


Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['./gguf-model_gguf/Qwen2.5-1.5B-Instruct.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions completed successfully!
Generated files: ['./gguf-model_gguf/Qwen2.5-1.5B-Instruct.Q4_K_M.gguf']
Unsloth: example usage for text only LLMs: /root/.unsloth/llama.cpp/llama-cli --model ./gguf-model_gguf/Qwen2.5-1.5B-Instruct.Q4_K_M.gguf -p "why is the sky blue?"
Unsloth: Saved Ollama Modelfile to ./gguf-model_gguf/Modelfile
Unsloth: convert model to ollama format by running - ollama create model_name -f ./gguf-model_gguf/Modelfile
✅ GGUF saved.


In [ ]:

# ── 12. Quick Inference Test ──────────────────────────────────
print("\n Running a quick inference test...")

FastLanguageModel.for_inference(model)   # enable 2× faster inference

sample_question = (
    "### Question:\n"
    "What is the difference between supervised and unsupervised learning?\n\n"
    "### Answer:\n"
)

inputs = tokenizer(sample_question, return_tensors="pt").to("cuda")
outputs = model.generate(
    **inputs,
    max_new_tokens  = 256,
    temperature     = 0.7,
    top_p           = 0.9,
    repetition_penalty = 1.1,
    use_cache       = True,
)
response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("\n── Sample Output ──────────────────────────────────────")
print(response[len(sample_question):])
print("────────────────────────────────────────────────────────\n")

print(" ALL DONE.")



🧪 Running a quick inference test...


Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12


── Sample Output ──────────────────────────────────────
Supervised learning involves training a model on labeled data, where each example in the dataset has a corresponding label. The goal of supervised learning is to learn a mapping from input features to output labels based on the labeled data.

Unsupervised learning, on the other hand, does not require labeled data. Instead, it uses unlabeled data to find patterns or structure within the data. The objective of unsupervised learning is to discover hidden structures or relationships in the data without any prior knowledge about the target variable.

In summary, supervised learning requires labeled data for training, while unsupervised learning works with unlabeled data. Supervised learning aims to predict the output given input features, whereas unsupervised learning seeks to uncover underlying patterns or groups within the data.
────────────────────────────────────────────────────────

✅ ALL DONE.


## Load and test GGUF Model

In [ ]:
!pip install llama-cpp-python

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.6/67.6 MB 11.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 5.7 MB/s eta 0:00:00
  Created wheel for llama-cpp-python: filename=llama_cpp_python-0.3.21-py3-none-linux_x86_64.whl size=18292438 sha256=b83af2455ce7077247c3cfe32bcfa820fd4fbed4d3d080b75e9ae3174a1c6268
  Stored in directory: /root/.cache/pip/wheels/8d/9c/44/39cb3a9e47ced67e64197053dace3cec12faf53daf81cac6c4
Successfully built llama-cpp-python


In [ ]:
from llama_cpp import Llama
from unsloth import FastLanguageModel
import torch

# 1) Load GGUF model
GGUF_PATH = "./gguf-model_gguf/Qwen2.5-1.5B-Instruct.Q4_K_M.gguf"

print(" Loading GGUF model...")
gguf_model = Llama(
    model_path=GGUF_PATH,
    n_ctx=2048,
    n_threads=4,
    n_gpu_layers=-1
)
print(" GGUF loaded\n")


# 2) Load BASE model (Unsloth)
print(" Loading BASE model...")

base_model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    load_in_4bit=True,
)

FastLanguageModel.for_inference(base_model)

print(" Base model loaded\n")


# Questions (2 examples)
questions = [
    "What is the difference between supervised and unsupervised learning?",
    "Explain overfitting in machine learning."
]


# Run comparison
for i, q in enumerate(questions):

    print("\n" + "="*70)
    print(f" EXAMPLE {i+1}")
    print("="*70)

    prompt = f"""### Question:
{q}

### Answer:
"""


    #  GGUF OUTPUT
    print("\n GGUF MODEL OUTPUT:\n")

    gguf_out = gguf_model(
        prompt,
        max_tokens=200,
        temperature=0.7,
        top_p=0.9,
        repeat_penalty=1.1
    )

    print(gguf_out["choices"][0]["text"])


    #  BASE MODEL OUTPUT
    print("\n BASE MODEL OUTPUT:\n")

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    outputs = base_model.generate(
        **inputs,
        max_new_tokens=200,
        temperature=0.7,
        top_p=0.9,
        repetition_penalty=1.1,
    )

    text = tokenizer.decode(outputs[0], skip_special_tokens=True)

    print(text[len(prompt):])


print("\n COMPARISON DONE (2 EXAMPLES)")

📦 Loading GGUF model...


llama_model_loader: loaded meta data with 25 key-value pairs and 338 tensors from ./gguf-model_gguf/Qwen2.5-1.5B-Instruct.Q4_K_M.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = qwen2
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = Gguf Model
llama_model_loader: - kv   3:                       general.quantized_by str              = Unsloth
llama_model_loader: - kv   4:                         general.size_label str              = 1.5B
llama_model_loader: - kv   5:                           general.repo_url str              = https://huggingface.co/unsloth
llama_model_loader: - kv   6:                               general.tags arr[str,2]       = ["unsloth", "llama.cpp"]
llama

✅ GGUF loaded

📦 Loading BASE model...
==((====))==  Unsloth 2026.4.8: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
✅ Base model loaded


🧪 EXAMPLE 1

🟡 GGUF MODEL OUTPUT:



llama_perf_context_print:        load time =    1562.10 ms
llama_perf_context_print: prompt eval time =    1561.75 ms /    18 tokens (   86.76 ms per token,    11.53 tokens per second)
llama_perf_context_print:        eval time =   45835.97 ms /   199 runs   (  230.33 ms per token,     4.34 tokens per second)
llama_perf_context_print:       total time =   48395.02 ms /   217 tokens
llama_perf_context_print:    graphs reused =        198
Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`

Supervised learning and unsupervised learning are two types of machine learning algorithms used for pattern recognition.

**Supervised Learning:**

- **Definition**: Supervised learning involves training a model on labeled data, where the input is known (features) and the output is also known (labels). The goal is to predict the correct label for new, unseen data.
  
  - **Example**: Predicting customer churn in a telecommunications company based on historical data.

**Unsupervised Learning:**

- **Definition**: Unsupervised learning involves training a model on unlabeled data, where the input features are not labeled. The goal is to find patterns or structure within the data without any predefined labels.
  
  - **Example**: Clustering customers into different segments based on their spending habits and preferences.

In summary, supervised learning requires labeled data for training, while unsupervised learning works with unlabeled data to identify patterns or clusters. Both types of 

Llama.generate: 3 prefix-match hit, remaining 12 prompt tokens to eval
llama_perf_context_print:        load time =    1562.10 ms
llama_perf_context_print: prompt eval time =     655.52 ms /    12 tokens (   54.63 ms per token,    18.31 tokens per second)
llama_perf_context_print:        eval time =   34865.05 ms /   199 runs   (  175.20 ms per token,     5.71 tokens per second)
llama_perf_context_print:       total time =   36289.65 ms /   211 tokens
llama_perf_context_print:    graphs reused =        198
Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Overfitting in machine learning refers to a situation where a model is trained too well on the training data, resulting in high accuracy on that specific dataset but poor performance when tested on new, unseen data. This occurs because the model has learned the noise and outliers as part of its training process, leading it to perform exceptionally well on the training set but poorly on validation or test sets.

To avoid overfitting, techniques such as regularization (adding a penalty for complexity in the cost function), early stopping (stopping the training when performance starts decreasing), and data augmentation (increasing the size of the training dataset by generating new examples) can be employed. Additionally, selecting an appropriate model architecture or choosing hyperparameters wisely is crucial to prevent overfitting.

### Example:
Imagine you are building a machine learning model for image classification. If your model performs exceptionally well on images from a specific 